# Squares in Squares: a certified 22.2498 portfolio, verified in-notebook, and the method behind it

**Score 22.249870307** = Σ over n = 1…1000 of L<sub>n</sub>/√n − 1. The CSV holds the 548 puzzles where a tilted packing
beats the grid; the metric charges the other 452 n their grid score. The notebook re-verifies all 548 packings with the **official competition metric**, which uses interval arithmetic
and allows no tolerance. That makes the `submission.csv` in the output safe to submit or merge.

**Contents**
1. The problem and what the score rewards
2. Exact verification with the official metric
3. Anatomy of the score: where the 22.256 comes from
4. Gallery: small and large packings
5. How the portfolio was built: tilted-strip theory, a gadget bank, fault-line DP, exact cap design, GLS separation
6. Negative results: what did **not** work, with numbers
7. Credits
8. `submission.csv`

A companion notebook, *Squares in Squares: Advanced EDA of Packings*, studies the packings themselves:
- wasted area against the Erdős–Graham and Chung–Graham bounds;
- the strip law hidden in the tilt angles;
- contact graphs and loose squares;
- a ranking of the puzzles with the most remaining gain.

In [ ]:
import os, sys, glob, math, time, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.collections import PatchCollection, LineCollection
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False})

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    hits += glob.glob(os.path.join(os.environ.get('SIS_DATA', '.'), name))
    assert hits, f'{name} not found: attach the dataset {DATASET!r}'
    return hits[0]

DATASET = 'mdmuqtadirfuad/sis-certified-portfolio'
CSV = find('portfolio.csv')
df = pd.read_csv(CSV)
P = {int(n): g.sort_values('square_id')[['x', 'y', 'angle']].to_numpy(float) for n, g in df.groupby('n')}

def corners(p):
    """Corner coordinates, two (m, 4) arrays, of unit squares given as rows (x, y, angle in radians)."""
    c, s = np.cos(p[:, 2]) / 2, np.sin(p[:, 2]) / 2
    ux = np.stack([c - s, -c - s, -c + s, c + s], 1)
    uy = np.stack([s + c, -s + c, -s - c, s - c], 1)
    return p[:, :1] + ux, p[:, 1:2] + uy

def side(p):
    """Side of the smallest axis-parallel bounding square (translation invariant, as in the metric)."""
    cx, cy = corners(p)
    return max(np.ptp(cx), np.ptp(cy))

def tilt(p):
    """Tilt of each square in degrees, folded to [0, 45] (a square is invariant under quarter turns)."""
    a = np.mod(p[:, 2] + np.pi / 4, np.pi / 2) - np.pi / 4
    return np.degrees(np.abs(a))

TILT_EPS = 1e-3                                 # degrees: below this a square counts as axis-aligned

ns = np.array(sorted(P))                        # the scored puzzles: n = K^2 + d, 1 <= d <= d_max(K)
pos = lambda x: np.searchsorted(ns, x)          # n -> row of the per-puzzle arrays below
L = np.array([side(P[n]) for n in ns])
loss = L / np.sqrt(ns) - 1                      # per-puzzle score
K = np.floor(np.sqrt(ns) + 1e-9).astype(int)
d = ns - K ** 2
grid = np.ceil(np.sqrt(ns) - 1e-12)             # side of the trivial grid packing
grid_loss = grid / np.sqrt(ns) - 1
f = L - np.floor(L + 1e-12)                     # L = floor(L) + f
def grid_score(m):
    g = math.isqrt(m); g += g * g < m
    return g / math.sqrt(m) - 1
REST = sum(grid_score(m) for m in range(1, 1001) if m not in P)   # omitted n are charged the grid score
print(f'{len(df):,} squares in {len(P)} packed puzzles: {loss.sum():.9f}  + {1000 - len(P)} grid puzzles: {REST:.9f}'
      f'  = total {loss.sum() + REST:.9f}')

CMAP = plt.cm.cool
UPRIGHT = (0.84, 0.86, 0.90, 1.0)

def draw(ax, n, lw=0.3, title=True, edges=None, mark=None):
    """Draw packing n: grey = axis-aligned, colour = tilt (0-45 deg), dashed red = bounding square.
    edges: optional (i, j) index arrays of contacts to draw; mark: boolean mask of squares to outline in red."""
    p = P[n]; cx, cy = corners(p); x0, y0 = cx.min(), cy.min(); Ln = side(p); th = tilt(p)
    fc = [CMAP(t / 45) if t > TILT_EPS else UPRIGHT for t in th]
    polys = [Polygon(np.c_[cx[i] - x0, cy[i] - y0]) for i in range(len(p))]
    ax.add_collection(PatchCollection(polys, facecolor=fc, edgecolor='#333', linewidth=lw))
    if mark is not None and mark.any():
        ax.add_collection(PatchCollection([polys[i] for i in np.flatnonzero(mark)], facecolor='none',
                                          edgecolor='red', linewidth=1.6))
    if edges is not None:
        i, j = edges
        seg = np.stack([p[i, :2] - (x0, y0), p[j, :2] - (x0, y0)], 1)
        ax.add_collection(LineCollection(seg, colors='k', linewidths=0.6, alpha=0.7))
    ax.add_patch(Rectangle((0, 0), Ln, Ln, fill=False, ec='crimson', lw=0.9, ls='--'))
    ax.set_xlim(-0.02 * Ln, 1.02 * Ln); ax.set_ylim(-0.02 * Ln, 1.02 * Ln)
    ax.set_aspect('equal'); ax.axis('off')
    if title:
        ax.set_title(f'n = {n}   L = {Ln:.6f}\nscore {Ln / math.sqrt(n) - 1:.5f}   {(th > TILT_EPS).sum()} tilted', fontsize=8)

def tilt_colorbar(fig, axs):
    sm = plt.cm.ScalarMappable(cmap=CMAP, norm=plt.Normalize(0, 45))
    fig.colorbar(sm, ax=list(np.ravel(axs)), shrink=0.6, label='tilt (degrees)')

## 1. The problem

Pack n unit squares, each with its own rotation, into the smallest possible square of side L<sub>n</sub>.

**Score.** The area bound gives L<sub>n</sub> ≥ √n, so each puzzle scores L<sub>n</sub>/√n − 1 ≥ 0. The total is
summed over the scored puzzles, and lower is better.

**Notation.** Write L = K + f and n = K² + d.

- **Only 548 n need a packing.** A missing or infeasible puzzle is charged the grid score ⌈√n⌉/√n − 1, so the CSV
  lists just the **548 puzzles** where something beats the grid: n = K² + d with 1 ≤ d ≤ d<sub>max</sub>(K), where
  d<sub>max</sub> grows from 1 at K = 2 to 37 at K = 31 (n = 5 … 998). For the other 452 n, including the perfect
  squares (score 0), no packing better than the grid is known. Those 452 cost a fixed 7.268 for everyone; the race
  is over the 548.
- **The grid is the baseline.** As soon as d > 0 the grid needs side K + 1, so its loss is about 1/K just after
  every perfect square.
- **Tilting beats it.** Every good packing beats the grid by tilting strips of squares, so that a side of length
  K + f < K + 1 holds more squares than K + f upright rows could. Section 5 gives the one-line law behind this.
- **Hardening is required.** The metric checks overlap with interval arithmetic at 50+ digits. Touching is allowed;
  an overlap of 1e-15 is not. After any floating-point optimisation, a packing must be hardened: squares are pushed
  apart by a few ulps.

## 2. Exact verification with the official metric

The next cell runs `metric.evaluate_single_puzzle` on all 548 puzzles, in parallel on every CPU. This is the
competition's scorer, shipped unmodified in the dataset as `metric.py`. Each puzzle first goes through the same
pandas float64 round trip that Kaggle applies to a submitted CSV.

Set `FULL_CERT = False` for a quick check of n ≤ 150 only.

In [ ]:
METRIC = find('metric.py')
sys.path.insert(0, os.path.dirname(METRIC))
import metric
import multiprocessing as mp

FULL_CERT = True
text = df.drop(columns=['id']).to_csv(index=False)        # the float64 round trip the Kaggle scorer sees
puzzles = metric._parse_submission_csv(text)
if not FULL_CERT:
    puzzles = {n: r for n, r in puzzles.items() if n <= 150}

def evaluate(item):
    return metric.evaluate_single_puzzle(*item)

work = sorted(puzzles.items(), key=lambda kv: -kv[0])     # largest first: better load balance
t0 = time.time()
if 'fork' in mp.get_all_start_methods():
    with mp.get_context('fork').Pool(os.cpu_count()) as pool:
        res = pool.map(evaluate, work, chunksize=1)
else:
    res = [evaluate(w) for w in work]
cert = pd.DataFrame(res, columns=['n', 'score', 'L', 'reason']).set_index('n').sort_index()
ok = cert.reason == 'ok'
print(f'certified {ok.sum()} / {len(cert)} puzzles in {time.time() - t0:.0f} s on {os.cpu_count()} CPUs')
print(f'official certified total : {cert.score.sum():.9f} (packed)' + ('' if FULL_CERT else '   (n <= 150 only)'))
if FULL_CERT:
    print(f'+ grid score of the {1000 - len(cert)} omitted n: {REST:.9f}  =>  TOTAL {cert.score.sum() + REST:.9f}')
print(f'float total (this file)  : {loss[pos(cert.index)].sum():.9f}')
print(f'largest per-puzzle |certified - float| : {np.abs(cert.score.to_numpy() - loss[pos(cert.index)]).max():.1e}')
assert ok.all(), cert[~ok]

## 3. Anatomy of the score

The four panels below show:

- **Top left:** per-puzzle score on a log scale, against the grid. The grid's saw-tooth (a jump to about 1/K right
  after each perfect square) is flattened by roughly an order of magnitude.
- **Top right:** cumulative score.
- **Bottom left:** K·score on the (K, d) lattice. Scaling by K removes the natural 1/K decay, so rows can be compared.
  The left edge, d = 1…3, stays expensive in every row: just after a perfect square, a few extra squares cost
  almost a full strip.
- **Bottom right:** how the total splits across the position t = d/(2K+1) inside a block.

In [ ]:
t = d / (2 * K + 1)
fig, ax = plt.subplots(2, 2, figsize=(13, 8.6))
a = ax[0, 0]
a.semilogy(ns, np.where(grid_loss > 0, grid_loss, np.nan), color='#bbb', lw=0.8, label='grid packing (side ⌈√n⌉)')
a.semilogy(ns, np.where(loss > 1e-12, loss, np.nan), color='C0', lw=0.8, label='this portfolio')
a.set(xlabel='n', ylabel='L/√n − 1  (log)', title='Per-puzzle score vs the trivial grid'); a.legend()
a = ax[0, 1]
a.plot(ns, np.cumsum(loss), label=f'this portfolio ({loss.sum():.4f} on the 548 packed n)')
a.plot(ns, np.cumsum(grid_loss), color='#bbb', label=f'grid ({grid_loss.sum():.2f})')
a.set(xlabel='n', ylabel='cumulative score', title='Cumulative score'); a.legend()
a = ax[1, 0]
M = np.full((K.max() + 1, 2 * K.max() + 1), np.nan); M[K, d] = K * loss
im = a.imshow(M[1:], aspect='auto', origin='lower', cmap='magma_r', interpolation='nearest',
              extent=[-0.5, M.shape[1] - 0.5, 0.5, K.max() + 0.5])
fig.colorbar(im, ax=a, label='K · score'); a.grid(False)
a.set(xlabel='d   (n = K² + d)', ylabel='K', title='K · score on the (K, d) lattice')
a = ax[1, 1]
bins = np.linspace(0, 1, 21)
share = pd.Series(loss).groupby(np.digitize(t, bins[1:-1])).sum()
a.bar(bins[:-1] + 0.025, share.reindex(range(20), fill_value=0), width=0.045, color='C1')
a.set_xlim(0, 0.65)
a.set(xlabel='t = d / (2K + 1)   (position inside a block)', ylabel='score contributed',
      title='Where the total comes from')
plt.tight_layout(); plt.show()

tab = pd.DataFrame({'range': pd.cut(ns, range(0, 1001, 100)), 'score': loss, 'grid': grid_loss})
tab = tab.groupby('range', observed=True).agg(score=('score', 'sum'), mean=('score', 'mean'), grid=('grid', 'sum'))
tab['saved vs grid'] = tab.grid - tab.score
display(tab.round(5))

## 4. Gallery

Grey squares are axis-aligned; coloured squares are tilted, with the colour giving the tilt angle (0°–45°). The
dashed red line is the bounding square.

**Small n.** These are the classical shapes: Göbel's 45° insert at n = 5, the n = 10 and n = 11 records, the
n = 17, 18 and 19 family, and the tilted cores of n = 26 … 52.

In [ ]:
small = [5, 10, 11, 17, 18, 19, 26, 27, 28, 37, 50, 52]
fig, axs = plt.subplots(3, 4, figsize=(13, 10.5))
for a, n in zip(axs.flat, small):
    draw(a, n, lw=0.6)
tilt_colorbar(fig, axs); plt.show()

**Large n.** Two architectures dominate.

- **Top row: 45° diamonds.** In each range, the packing that saves the most against the grid. These sit just above a
  perfect square: a rotated block whose diagonal fits a side K + f with f = √2/2, (5√2 − 6)/2 or 4√2 − 5. These are
  the three largest families in the EDA notebook.
- **Bottom row: strip systems.** In each range, the packing with the most squares tilted between 5° and 40°. These are
  upright blocks with tilted strips of thickness m + f along the sides, and the strip ends ("caps") are where the art
  is.

In [ ]:
mid = np.array([((tilt(P[n]) > 5) & (tilt(P[n]) < 40)).sum() for n in ns])   # squares in tilted strips
diamonds, strips = [], []
for lo, hi in [(100, 300), (300, 650), (650, 1000)]:
    r = (ns >= lo) & (ns <= hi)
    diamonds.append(int(ns[r][np.argmax((grid_loss - loss)[r])]))
    strips.append(int(ns[r][np.argmax(mid[r])]))
fig, axs = plt.subplots(2, 3, figsize=(15, 10.5))
for a, n in zip(axs.flat, diamonds + strips):
    draw(a, n, lw=0.15)
tilt_colorbar(fig, axs); plt.show()

## 5. How the portfolio was built

### 5.1 Tilted strips: the one idea behind every non-grid packing

Take a strip of height H = m + f along one side of the square. Instead of m upright rows, it holds columns of
**m + 1 squares tilted by θ**, where

$$(m+1)\cos\theta + \sin\theta = m + f .$$

**Gain per unit length.** The columns repeat with pitch 1/cos θ. Per unit of strip length, the strip therefore holds
(m+1)cos θ squares instead of m, a gain of

$$s(m, f) = f - \sin\theta \quad\text{squares per unit length.}$$

**Building a packing.** A side of length K + f carries such strips. A gain of s per unit length over about K units
of length buys about s·K extra squares. That gain is paid for at the strip ends ("caps"): in all our measurements a
cap costs about 0.32–0.38·H. Strips along two sides plus one upright block give the classic construction; the best
packings nest several strips and design the caps exactly.

**Reading the plot.** It solves the strip equation exactly for several thicknesses m. A strip only pays for itself
where s > 0, and thick strips (large m) need f close to 1.

In [ ]:
ff = np.linspace(0.001, 0.999, 600)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for m in [1, 2, 3, 5, 8, 12, 20]:
    R, phi = math.hypot(m + 1, 1), math.atan2(1, m + 1)
    th = phi + np.arccos((m + ff) / R)          # the root with theta > 0 of (m+1)cos(theta) + sin(theta) = m + f
    s = ff - np.sin(th)
    ax[0].plot(ff, np.degrees(th), label=f'm = {m}')
    ax[1].plot(ff, s, label=f'm = {m}')
    z = ff[np.argmax(s > 0)]
    ax[1].plot(z, 0, 'k.', ms=5)
ax[0].set(xlabel='f  (fractional part of the side)', ylabel='tilt θ (degrees)', title='Tilt of an (m+1)-column in height m + f')
ax[1].axhline(0, color='k', lw=0.6)
ax[1].set(xlabel='f', ylabel='s = f − sin θ', title='Extra squares per unit strip length (dots: break-even f)')
ax[0].legend(ncol=2); plt.tight_layout(); plt.show()

### 5.2 The pipeline

| # | stage | what it does |
|---|---|---|
| 1 | **Harvest** | Pull every public notebook output daily, reading **CSV only**. Harden each packing and keep it only if the official metric certifies it and it beats the incumbent by more than 1e-12. |
| 2 | **Gadget bank** | Cut every certified packing along straight lines into rectangular sub-packings ("gadgets": an a × b box at fractional part f, with its square count). The bank holds about 50,000 entries. |
| 3 | **Fault-line staircase DP** | Split a square of side K + f along monotone staircase fault lines into rectangles, then choose gadgets with the same f by DP to maximise the count. Reaching n with a smaller f than the incumbent gives a new packing. The pieces are certified already, so only the seams need re-checking. |
| 4 | **Exact cap design** | Strip ends are designed as a maximum independent set over contact-anchored candidate square positions (CP-SAT on Kaggle, Gurobi locally), maximising the squares a cap holds at a given strip phase. |
| 5 | **GLS separation** | A Sparrow-style guided local search (adaptive overlap weights, collision-focused moves) turns approximately feasible layouts into valid ones, followed by hardening. |
| 6 | **Certification** | Run the official metric on every changed puzzle, then on the full file before every submission. |

**Honest attribution.** Most of this portfolio is the community's public work, merged and re-certified. Our own
constructions (DP assemblies and designed caps) account for a few hundredths of the total. Merging every new public
notebook every day was the single most valuable habit.

**Compute.** All research experiments ran as Kaggle CPU kernels; no GPU was used. One practical tip: set
`OMP_NUM_THREADS` = `OPENBLAS_NUM_THREADS` = `NUMBA_NUM_THREADS` = 1 *before* importing numpy in multiprocess
kernels. That gave us a 45× speed-up on Kaggle CPUs.

## 6. What did **not** work

Every item below was tested on Kaggle CPU; the numbers come from our research log.

- **Insert and repair.** We added 1–2 squares to a tight packing (n 590 → 591 / 592) and re-separated with GLS and
  with evand's `fq` finisher: **0 successes in 24 tries**. The control run, re-compressing n 590 itself, passes. Local
  repair cannot create the room a new square needs.
- **Cap cost is stubborn.** Strip ends cost **0.32–0.38 H** at every phase we measured. Two predictive models, wall
  frustration and taper, failed out of sample: 0.465 H measured against 0.29 H predicted at m = 9.
- **Commensurate strips.** No leaderboard packing contains an exact rational-slope tilted strip (0 matches at 1e-9).
- **"Graded micro-tilt fans"** seen in some record packings are rattlers, not structure.
- **Intermediate angles.** Offering fan or 45° candidates to the exact cap designer never changes its optimum.
- **Bigger rooms.** We tried to fit 98 squares in a 5.799 × 18.799 box and 122 in a 19.799 × 6.799 box with one
  corner free: **0 / 16** GLS seeds each.

**Lesson:** the gap at the top of the leaderboard is *architecture* (how strips, blocks and caps are arranged
globally), not local polishing.

## 7. Credits

This portfolio stands on the community's shoulders. Packings were merged, after re-certification, from public
notebook outputs by (most merged first):

nilanjansaha123, prvsiyan, alexchilton, malyshevdanil, lucifer19, busyaprime, seshurajup, saamhm, lianlian99,
avazaku, sigmaborov, inversion, rushikeshpundkar, myadiamond, kirilltym, citerne, chaitanyajamble, aqibrazadev,
amirhezekiahespanto and many more, plus the early fault-line DP, SquareCraft, ensemble and per-puzzle-leaderboard
notebooks, Chris Deotte's strip notebooks, evand's open `square-packing` code, and the per-puzzle certificates
collected from the discussion forum.

The classical background comes from:
- Erich Friedman's *Packing Center*;
- Erdős & Graham (1975), Roth & Vaughan (1978) and Chung & Graham (2020) on wasted area.

Thank you all. If one of your packings is in here and you are not named, tell me in the comments and I will add you.

## 8. `submission.csv`

The certified file, copied byte for byte, so the decimal expansions the metric verified are exactly what gets
submitted.

In [ ]:
out = '/kaggle/working/submission.csv' if os.path.isdir('/kaggle/working') else 'submission.csv'
shutil.copyfile(CSV, out)
print('wrote', out, f'{os.path.getsize(out) / 2**20:.1f} MB')